# EpiLink interactive heatmap walkthrough

This notebook focuses on one interactive view: a compatibility heatmap with sampling interval on the x-axis and genetic distance on the y-axis.

Use the controls to switch between:

- target subsets
- ancestor-descendant scenarios `ad(m)`
- common-ancestor scenarios `ca(m_i,m_j)`

Run this notebook in the `epilink` conda environment so that `epilink`, `scipy`, and `ipywidgets` are available.

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from epilink import EpiLink, InfectiousnessToTransmission, NaturalHistoryParameters

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (8, 5),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "axes.labelsize": 12,
        "figure.titlesize": 18,
    }
)

rng_seed = 2026


## Build an EpiLink model

We use the infectiousness-to-transmission profile and precompute Monte Carlo draws for a modest hidden-depth limit.

In [2]:
parameters = NaturalHistoryParameters()
profile = InfectiousnessToTransmission(parameters=parameters, rng_seed=rng_seed)

epilink = EpiLink(
    transmission_profile=profile,
    maximum_depth=3,
    mc_samples=8000,
    target=["ad(0)", "ca(0,0)"],
    mutation_process="stochastic",
)

print("Configured target subset:", epilink.target_labels)
print("Available scenarios:", [scenario.label() for scenario in epilink.scenarios])


Configured target subset: ('ad(0)', 'ca(0,0)')
Available scenarios: ['ad(0)', 'ad(1)', 'ad(2)', 'ad(3)', 'ca(0,0)', 'ca(0,1)', 'ca(0,2)', 'ca(0,3)', 'ca(1,0)', 'ca(1,1)', 'ca(1,2)', 'ca(2,0)', 'ca(2,1)', 'ca(3,0)']


## Interactive compatibility heatmap

The heatmap always uses:

- x-axis: sample-time difference `Delta t`
- y-axis: genetic distance `g`

Then the controls switch which scenario or target subset is being scored.

In [3]:
import ipywidgets as widgets
from IPython.display import display

time_grid = np.linspace(-2, 20, 160)
genetic_grid = np.linspace(0, 10, 160)
T, G = np.meshgrid(time_grid, genetic_grid)

ad_labels = [scenario.label() for scenario in epilink.scenarios if scenario.kind == "ad"]
ca_labels = [scenario.label() for scenario in epilink.scenarios if scenario.kind == "ca"]
all_labels = [scenario.label() for scenario in epilink.scenarios]

target_models = {
    "Configured target subset": epilink.pairwise_model(),
    "All AD scenarios": epilink.pairwise_model(ad_labels),
    "All CA scenarios": epilink.pairwise_model(ca_labels),
    "All scenarios": epilink.pairwise_model(all_labels),
}
ad_models = {label: epilink.pairwise_model(label) for label in ad_labels}
ca_models = {label: epilink.pairwise_model(label) for label in ca_labels}

surface_cache = {}


def get_surface(cache_key, model):
    if cache_key not in surface_cache:
        surface_cache[cache_key] = model(T, G)
    return surface_cache[cache_key]


view = widgets.ToggleButtons(
    options=[
        ("Target subset", "target"),
        ("AD scenario", "ad"),
        ("CA scenario", "ca"),
    ],
    description="View:",
)

target_choice = widgets.Dropdown(
    options=list(target_models.keys()),
    value="Configured target subset",
    description="Target:",
    layout=widgets.Layout(width="420px"),
)

ad_choice = widgets.SelectionSlider(
    options=ad_labels,
    value=ad_labels[0],
    description="AD:",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)

ca_choice = widgets.SelectionSlider(
    options=ca_labels,
    value=ca_labels[0],
    description="CA:",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)

output = widgets.Output()


def sync_controls():
    target_choice.layout.display = "" if view.value == "target" else "none"
    ad_choice.layout.display = "" if view.value == "ad" else "none"
    ca_choice.layout.display = "" if view.value == "ca" else "none"


def draw_heatmap(*_):
    sync_controls()

    if view.value == "target":
        label = target_choice.value
        model = target_models[label]
        colorbar_label = "Compatibility (summed across scenarios)"
    elif view.value == "ad":
        label = ad_choice.value
        model = ad_models[label]
        colorbar_label = "Compatibility"
    else:
        label = ca_choice.value
        model = ca_models[label]
        colorbar_label = "Compatibility"

    surface = get_surface((view.value, label), model)

    with output:
        output.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7.8, 5.3), constrained_layout=True)
        image = ax.imshow(
            surface,
            origin="lower",
            extent=[time_grid.min(), time_grid.max(), genetic_grid.min(), genetic_grid.max()],
            aspect="auto",
            cmap="magma",
        )
        contours = ax.contour(
            time_grid,
            genetic_grid,
            surface,
            levels=6,
            colors="white",
            linewidths=0.7,
            alpha=0.8,
        )
        ax.clabel(contours, inline=True, fontsize=8, fmt="%.2f")
        ax.set(
            title=f"Compatibility heatmap: {label}",
            xlabel="Sample-time difference (days)",
            ylabel="Genetic distance",
        )
        fig.colorbar(image, ax=ax, shrink=0.92, label=colorbar_label)
        plt.show()


for control in (view, target_choice, ad_choice, ca_choice):
    control.observe(draw_heatmap, names="value")

sync_controls()
display(widgets.VBox([view, target_choice, ad_choice, ca_choice, output]))
draw_heatmap()

## Next steps

- Increase `maximum_depth` to expose deeper latent scenarios.
- Switch between deterministic and stochastic mutation processes.
- Replace the configured target subset with the scenario family you want to study.
- Save selected surfaces if you want static figures for the manuscript.